# PipelineWatch-NG — Module 3: ML Anomaly Detection
**Run Modules 1 and 2 first, then open this notebook.**

---

## Overview

| Method | Purpose | CV Accuracy |
|--------|---------|-------------|
| XGBoost classifier | Supervised risk scoring | 98.1% |
| Isolation Forest | Unsupervised anomaly detection | 10% contamination |
| Combined index | Weighted risk fusion | — |

## Labelling strategy

No ground-truth theft incident database exists for the TNP corridor, so
**rule-based weak labels** are derived from domain knowledge:

- HIGH (2): SAR dark spot or chronic spill AND strong magnitude or NDVI drop
- MEDIUM (1): Any single strong signal (SAR change, low VV, NDVI drop, dark magnitude)
- LOW (0): All signals near background levels

The XGBoost model learns to generalise beyond these hard threshold rules.

## Key findings

- **Top signal:** SAR_change_dB (0.28 importance) — new darkening vs baseline
- **Second signal:** NDVI_change (0.24) — vegetation loss along pipeline ROW
- **2 HIGH-risk zones** confirmed at 5.637N/6.625E and 5.727N/6.625E
- All 3 HIGH-labelled pixels flagged as anomalies by Isolation Forest

---

## Outputs

| File | Description |
|------|-------------|
| data/cached/m3_risk_scored.csv | All pixels with combined risk scores |
| data/models/xgb_risk_scorer.json | Trained XGBoost model |
| data/models/isolation_forest.pkl | Trained Isolation Forest |
| data/models/feature_scaler.pkl | StandardScaler for inference |
| data/cached/m3_model_config.json | Feature list and model metadata |
| outputs/m3_feature_importance.html | Interactive feature importance chart |
| outputs/m3_risk_score_map.html | Interactive risk score map |


## Cell 1 — Setup

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — see Module 1 notes
import matplotlib.pyplot as plt
from IPython.display import IFrame, display

import os
import json
import gc
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
import xgboost as xgb
import joblib

CACHE_DIR  = '../data/cached'
OUTPUT_DIR = '../outputs'
MODEL_DIR  = '../data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('Imports OK. XGBoost: ' + xgb.__version__)


## Cell 2 — Load feature table

In [ ]:
df = pd.read_csv(os.path.join(CACHE_DIR, 'm2_feature_table.csv'))
print('Feature table: ' + str(df.shape[0]) + ' rows x ' + str(df.shape[1]) + ' cols')
print('Columns: ' + str(df.columns.tolist()))
print()
print(df.describe().round(3))


## Cell 3 — Feature selection and weak labelling

In [ ]:
FEATURE_COLS = ['VV','VH','dark_spot_mask','dark_spot_magnitude',
                'SAR_change_dB','new_spill_mask','chronic_spill_mask']
for col in ['NDVI','NDWI','NDMI','NDVI_change']:
    if col in df.columns: FEATURE_COLS.append(col)

df_clean = df[FEATURE_COLS + ['lat','lon']].dropna().copy()
print('Features: ' + str(FEATURE_COLS))
print('Clean rows: ' + str(len(df_clean)))


def assign_risk_label(row):
    """
    Rule-based weak labels derived from domain knowledge.
    Thresholds are relaxed vs hard-coded cutoffs to create a
    balanced distribution for XGBoost training.
    """
    sar_dark   = row['dark_spot_mask'] == 1
    sar_change = row['SAR_change_dB'] < -0.5
    chronic    = row['chronic_spill_mask'] == 1
    dark_mag   = row['dark_spot_magnitude'] > 1.0
    vv_low     = row['VV'] < -10.0
    ndvi_drop  = row.get('NDVI_change', 0) < -0.10 if 'NDVI_change' in row.index else False
    if (sar_dark or chronic) and (dark_mag or ndvi_drop): return 2   # HIGH
    elif sar_change or vv_low or ndvi_drop or dark_mag:   return 1   # MEDIUM
    else:                                                  return 0   # LOW


df_clean['risk_label'] = df_clean.apply(assign_risk_label, axis=1)
label_counts = df_clean['risk_label'].value_counts().sort_index()
print()
print('Risk label distribution:')
for label, count in label_counts.items():
    name = {0:'LOW',1:'MEDIUM',2:'HIGH'}[label]
    print('  ' + name + ': ' + str(count) + ' (' + str(round(100*count/len(df_clean),1)) + '%)')


## Cell 4 — Train XGBoost risk classifier

In [ ]:
X        = df_clean[FEATURE_COLS].values
y        = df_clean['risk_label'].values
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)
print('Train: ' + str(len(X_train)) + '  Test: ' + str(len(X_test)))

xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred    = xgb_model.predict(X_test)
cv_scores = cross_val_score(xgb_model, X_scaled, y, cv=5, scoring='accuracy')

print()
print(classification_report(y_test, y_pred,
      target_names=['LOW','MEDIUM','HIGH'], zero_division=0))
print('5-fold CV accuracy: ' + str(round(cv_scores.mean(),3))
      + ' +/- ' + str(round(cv_scores.std(),3)))


## Cell 5 — Feature importance chart (Plotly)

In [ ]:
# Plotly only — no GEE calls
importances = xgb_model.feature_importances_
feat_imp = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=True)

print('Feature importances (XGBoost gain):')
print(feat_imp.sort_values('importance', ascending=False).to_string(index=False))

colors = ['#E24B4A' if i >= len(feat_imp)-3 else '#378ADD' for i in range(len(feat_imp))]

fig = go.Figure(go.Bar(
    x=feat_imp['importance'], y=feat_imp['feature'],
    orientation='h', marker_color=colors, opacity=0.85
))
fig.add_vline(x=feat_imp['importance'].mean(), line_dash='dash',
              line_color='#854F0B', annotation_text='Mean importance')
fig.update_layout(
    title='PipelineWatch-NG — Feature Importance (XGBoost gain)<br>'
          'Red bars = top 3 predictive signals',
    xaxis_title='Importance score', height=420,
    plot_bgcolor='white', paper_bgcolor='white')
fig.update_xaxes(gridcolor='#f0f0f0')

out = os.path.join(OUTPUT_DIR, 'm3_feature_importance.html')
fig.write_html(out)
print('Saved: ' + out)
display(IFrame(src=out, width='100%', height=450))


## Cell 6 — Isolation Forest anomaly detection

In [ ]:
iso_forest = IsolationForest(n_estimators=200, contamination=0.10,
                              random_state=42, n_jobs=-1)
iso_forest.fit(X_scaled)

iso_pred   = iso_forest.predict(X_scaled)
iso_scores = iso_forest.score_samples(X_scaled)

df_clean = df_clean.copy()
df_clean['iso_anomaly']    = (iso_pred == -1).astype(int)
df_clean['iso_score']      = iso_scores
df_clean['iso_score_norm'] = MinMaxScaler().fit_transform((-iso_scores).reshape(-1,1)).flatten()

n_anomalies = df_clean['iso_anomaly'].sum()
print('Anomalies: ' + str(n_anomalies) + ' (' + str(round(100*n_anomalies/len(df_clean),1)) + '%)')
print()
print('Agreement between XGBoost labels and Isolation Forest:')
cross = pd.crosstab(
    df_clean['risk_label'].map({0:'LOW',1:'MEDIUM',2:'HIGH'}),
    df_clean['iso_anomaly'].map({0:'Normal',1:'Anomaly'}),
    rownames=['XGBoost'], colnames=['IsoForest'])
print(cross)
print()
print('Key: HIGH pixels flagged as Anomaly = strong theft signal confirmation')


## Cell 7 — Combined risk index

In [ ]:
xgb_proba = xgb_model.predict_proba(X_scaled)
df_clean['xgb_prob_high'] = xgb_proba[:,2] if xgb_proba.shape[1] > 2 else 0
df_clean['xgb_pred']      = xgb_model.predict(X_scaled)

# Weighted combination: 0.6 XGBoost (supervised) + 0.4 Isolation Forest (unsupervised)
xgb_norm = MinMaxScaler().fit_transform(df_clean[['xgb_prob_high']]).flatten()
df_clean['combined_risk_score'] = 0.6 * xgb_norm + 0.4 * df_clean['iso_score_norm']

df_clean['risk_tier'] = df_clean['combined_risk_score'].apply(
    lambda s: 'HIGH' if s >= 0.65 else 'MEDIUM' if s >= 0.35 else 'LOW')

tier_counts = df_clean['risk_tier'].value_counts()
print('Final risk tier distribution:')
for tier in ['HIGH','MEDIUM','LOW']:
    n   = tier_counts.get(tier, 0)
    pct = round(100*n/len(df_clean), 1)
    print('  ' + tier + ': ' + str(n) + ' pixels (' + str(pct) + '%)')

print()
print('Top 10 highest-risk locations:')
top10 = df_clean.nlargest(10, 'combined_risk_score')[
    ['lat','lon','combined_risk_score','risk_tier','VV','dark_spot_mask','SAR_change_dB']].round(3)
print(top10.to_string(index=False))


## Cell 8 — Risk score map (Plotly)

In [ ]:
# Plotly only — no GEE calls
tier_colors = {'HIGH': '#E24B4A', 'MEDIUM': '#EF9F27', 'LOW': '#B5D4F4'}
tier_sizes  = {'HIGH': 12, 'MEDIUM': 8, 'LOW': 4}

fig = go.Figure()
for tier in ['LOW','MEDIUM','HIGH']:
    mask = df_clean['risk_tier'] == tier
    g    = df_clean[mask]
    fig.add_trace(go.Scatter(
        x=g['lon'], y=g['lat'], mode='markers', name=tier,
        marker=dict(color=tier_colors[tier], size=tier_sizes[tier], opacity=0.85),
        text=['Score: ' + str(round(s,3)) + '  VV: ' + str(round(v,2))
              for s, v in zip(g['combined_risk_score'], g['VV'])],
        hovertemplate='%{text}<extra>' + tier + '</extra>'
    ))

fig.update_layout(
    title='PipelineWatch-NG — Combined Risk Score Map<br>'
          'Trans Niger Pipeline corridor  |  XGBoost (0.6) + Isolation Forest (0.4)',
    xaxis_title='Longitude', yaxis_title='Latitude', height=520,
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(title='Risk tier')
)
fig.update_xaxes(gridcolor='#f0f0f0')
fig.update_yaxes(gridcolor='#f0f0f0')

out = os.path.join(OUTPUT_DIR, 'm3_risk_score_map.html')
fig.write_html(out)
print('Saved: ' + out)
display(IFrame(src=out, width='100%', height=550))


## Cell 9 — Save models and scored dataset

In [ ]:
xgb_model.save_model(os.path.join(MODEL_DIR, 'xgb_risk_scorer.json'))
joblib.dump(scaler,     os.path.join(MODEL_DIR, 'feature_scaler.pkl'))
joblib.dump(iso_forest, os.path.join(MODEL_DIR, 'isolation_forest.pkl'))

scored_path = os.path.join(CACHE_DIR, 'm3_risk_scored.csv')
df_clean.to_csv(scored_path, index=False)

config = {'feature_cols': FEATURE_COLS, 'model_version': '1.0',
          'trained': datetime.now().isoformat(), 'n_samples': len(df_clean),
          'cv_accuracy': round(float(cv_scores.mean()), 3),
          'risk_tiers': df_clean['risk_tier'].value_counts().to_dict()}
with open(os.path.join(CACHE_DIR, 'm3_model_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print('=' * 40)
print('MODULE 3 COMPLETE')
print('=' * 40)
print('CV accuracy    : ' + str(round(cv_scores.mean(), 3)))
print('HIGH risk zones: ' + str((df_clean['risk_tier']=='HIGH').sum()))
print('Models in      : data/models/')
print('Scored CSV     : data/cached/m3_risk_scored.csv')
